# 第67课 · 量化「听错了多少」——从零实现编辑距离（edit distance），WER 的数学地基

**学习目标**
1. 理解编辑距离的三种操作：插入、删除、替换
2. 从零实现 O(m×n) 动态规划算法
3. 用它实现 WER（词错误率）核心计算
4. 明白 ASR 评估为什么离不开它

> **算法课**：小字符串手填 DP 表，再写代码。

← **上一课**　[L66 · ASR 系统全览](L66_asr_overview.ipynb)

> 上节课学习了 **ASR 系统全览**：声学模型 → 语言模型 → 解码器，端到端 vs 经典流水线。  
> 本课将探讨 **Edit Distance 从零实现**。

## 本课剧情：Google 翻译怎么知道你打了多少个错别字？

你在搜索框里打了"teh cat sat"，Google 立刻建议"the cat sat"。背后用的是什么算法？

**编辑距离（Edit Distance）**：把一个字符串改成另一个，最少需要几次操作？操作只有三种：

| 操作 | 例子 | 成本 |
|---|---|---|
| 替换（Substitution S） | `cat` → `cut`（a→u）| 1 |
| 插入（Insertion I） | `cat` → `cats`（加s）| 1 |
| 删除（Deletion D） | `cats` → `cat`（去s）| 1 |

注意：`teh` → `the` 的编辑距离是 **2**（e→h、h→e 两次替换）——标准 Levenshtein 没有"相邻交换"这种操作。

**为什么标准 Levenshtein 不允许交换？** 看起来"交换e和h"是1次操作，比2次替换更便宜，但历史上 Levenshtein 的定义就选了"删除、插入、替换"这三个。如果加上交换，会变成 **Damerau-Levenshtein 距离**——这是另一个算法，用在拼写检查；但语音识别的标准评估（WER）一直用原版，所以我们这里也遵循原版。记住：两个字符串对齐，最多只能通过替换来"纠正"一个位置上的差异，没有"跨位置的交换"。

**关键洞察**：Levenshtein 动态规划把"两个字符串的对齐问题"分解成**子问题**：

```
dp[i][j] = 将 a[:i] 变成 b[:j] 的最小操作数

if a[i-1] == b[j-1]:  dp[i][j] = dp[i-1][j-1]        # 字符相同，不需操作
else:                  dp[i][j] = 1 + min(
                           dp[i-1][j],    # 删除 a[i-1]
                           dp[i][j-1],    # 插入 b[j-1]
                           dp[i-1][j-1]   # 替换
                       )
```

**Aurora 连接**：`aurora.speech` 的 WER 评估工具正是这个算法的词级版本，用于量化 ASR 系统的输出质量。

本节任务：先实现字符级 `edit_distance(a, b)`，再组合成词级 `wer(reference, hypothesis)`。

In [ ]:
import numpy as np

## 💭 为什么这个 DP 能保证最优？

看到 `dp[i][j] = 1 + min(三个邻居)` 时，你可能会问：**这样贪心选最优的子问题，怎么保证整体是最优的？**

**直观理由（不需要正式证明）**：

编辑距离问题具有"**最优子结构**"——任何两个字符串的最短编辑路径，都可以分解成**更短字符串的最短编辑路径** + **最后一步操作**。

比如把 `'kitten'` 改成 `'sitting'`：
- 最短路径必然是这样的：先把 `'kitte'` 改成某个 `'sittin'` 的前缀（用最少操作），再处理最后一个字符 n 和 g
- 或者：先把 `'kitten'` 的前缀改成 `'sittin'`，再追加或删除末尾

在每一步，我们**只需要知道当前三个邻居中哪一个代价最小**——因为无论选哪条邻居到达 `dp[i][j]`，后续的最优解都不会改变。这叫"**贪心选择性质**"与"**最优子结构**"的结合，正是动态规划的核心。

**更严格的论证**（可选）：假设有某条路径不选 `min()` 中的最优邻居，而选了较差的邻居，到达 `dp[i][j]` 时成本就已经更大。之后到达终点 `dp[m][n]` 的总成本也必然更大（因为不能"反向优化"），所以这条路径不可能是全局最优。反证法：只有 `min()` 的选择能拼出全局最优路径。

记住：**DP 表的每一格都是"从开始到这里的最优成本"**，依次填满全表后，右下角 `dp[m][n]` 就是整体的全局最优。

## 🔬 动手前先填一格：DP 表到底怎么长出来

上面 `dp[i][j] = 1 + min(三个邻居)` 抽象，第一次很难落地。用最小例子 `a="ab"` → `b="ac"` 亲手填一格，公式立刻具象。

**先理解坐标系统**：
- `dp[i][j]` 代表的含义：**把 `a` 的前 `i` 个字符改成 `b` 的前 `j` 个字符的最少操作数**
- 因此 `dp[0][j]` 对应从**空字符串**变成 `b` 的前 `j` 个字符（只能通过 `j` 次**插入**完成）
- `dp[i][0]` 对应从 `a` 的前 `i` 个字符变成**空字符串**（只能通过 `i` 次**删除**完成）
- 这就是为什么 `dp[i][0] = i` 和 `dp[0][j] = j`

所以我们需要 **(m+1) × (n+1)** 的表（而不是 m×n），因为第 0 行和第 0 列是**边界条件**——它们代表与空字符串的转换。

`dp[i][j]` = 把 `a` 前 `i` 个字符改成 `b` 前 `j` 个字符的最少操作数：

```
        ""   a    c
   ""    0    1    2      ← 第 0 行 dp[0][j]=j（空串靠插入变出 j 个字符）
   a     1    0    1
   b     2    1  ┌─?─┐    ← dp[2][2] 待填
                 └───┘
```

**填 `dp[2][2]`** 时，我们在处理 `a[1]='b'` 和 `b[1]='c'`，它们**不相同**。此时有三个邻居可以考虑：

| 邻居位置 | 含义 | 公式 | 值 |
|---|---|---|---|
| `dp[1][1]`（左上） | 已处理 a 的前 1 个 + b 的前 1 个，现在用**替换**把 a[1]('b') 改成 b[1]('c') | `dp[1][1] + 1` | `0+1 = 1` |
| `dp[2][1]`（左） | 已处理 a 的全部 2 个 + b 的前 1 个，现在**插入** b[1]('c') 到 a 末尾 | `dp[2][1] + 1` | `1+1 = 2` |
| `dp[1][2]`（上） | 已处理 a 的前 1 个 + b 的全部 2 个，现在**删除** a[1]('b') | `dp[1][2] + 1` | `1+1 = 2` |

取最小 → `dp[2][2] = 1`（一次替换搞定）。

**核心规律**（记住这个"手势"）：
- **字符相同**（`a[i-1] == b[j-1]`）：直接抄左上角 `dp[i-1][j-1]`、不 +1（因为不需要操作）
- **字符不同**（`a[i-1] != b[j-1]`）：看三个邻居 +1，取最小
  - 左上 `dp[i-1][j-1] + 1` → **替换** a[i-1] 成 b[j-1]
  - 左 `dp[i][j-1] + 1` → **插入** b[j-1] 到 a 的处理位置后
  - 上 `dp[i-1][j] + 1` → **删除** a[i-1]

这三个邻居的含义为什么这样对应？**简单记法**：
- **左**（`i` 不变，`j` 增加）：处理 b 的更多字符，意味着在 a 的当前位置**后面插入** b 的字符
- **上**（`i` 增加，`j` 不变）：处理 a 的更多字符但 b 的进度不变，意味着**删除** a 中多处理的字符
- **左上**（`i` 增加，`j` 增加）：两边都往前走，意味着这两个位置的字符通过**替换**对齐

## 💡 关键问题：为什么字符相同时直接抄左上角？

有人会问：**如果 `a[i-1] == b[j-1]`，为什么一定要选 `dp[i-1][j-1]` 而不考虑其他邻居？**

**答案**：因为"直接不操作"严格优于所有替代方案。

想象你要把 `'cat'` 改成 `'cat'`：
- 选择"不操作"：`0` 步
- 选择"先删后插"（替代）：删 'c'，再插 'c' = `2` 步 ⟹ 肯定不如直接不操作
- 任何其他绕路都要额外成本

数学上：当 `a[i-1] == b[j-1]` 时，
```
dp[i-1][j-1]                                       ✓ 最优
1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])    ✗ 必然 ≥ 1 + dp[i-1][j-1]
```

所以直接抄左上角 **不仅可以**，**而且必须**——任何其他选择都会加冤枉的成本。这就叫"**最优子结构**"的体现。

## ✏️ 任务 1：实现字符级别编辑距离

**实现规范**：

```python
def edit_distance(a, b) -> int:
    # 初始化 (len(a)+1) × (len(b)+1) 的 dp 表
    # dp[i][0] = i（删除 a 的前 i 个字符）
    # dp[0][j] = j（插入 b 的前 j 个字符）
    # 按行填表，返回 dp[m][n]
```

**验收标准**：

| 输入 | 期望输出 | 原因 |
|---|---|---|
| `('', '')` | `0` | 空→空，0步 |
| `('abc', '')` | `3` | 3次删除 |
| `('abc', 'abc')` | `0` | 完全相同 |
| `('kitten', 'sitting')` | `3` | 经典案例 |
| `('', 'abc')` | `3` | 3次插入 |

**注意**：函数接受 `str` 或 `list`（后续 `wer()` 会传词列表）。

### 推导示范：`"kitten"` → `"sitting"`

这是编辑距离的经典例子，结果是 3。为了帮助你对照思路，下面给出 **DP 表的前 3 行**（对应 `a` 的前 3 个字符：k、i、t）：

```
        ""   s    i    t    t    i    n    g
   ""    0    1    2    3    4    5    6    7
   k     1    1    2    3    4    5    6    7
   i     2    2    1    2    3    4    5    6
   t     3    3    2    1    2    3    4    5
```

**手推第 1 行**（a = "k"，逐列填充）：
- `dp[1][0] = 1`（初始化：删除 k）
- `dp[1][1]`：`a[0]='k'` vs `b[0]='s'` 不同 → `1 + min(dp[0][1]=1, dp[1][0]=1, dp[0][0]=0) = 1`（替换）
- `dp[1][2]`：`a[0]='k'` vs `b[1]='i'` 不同 → `1 + min(dp[0][2]=2, dp[1][1]=1, dp[0][1]=1) = 2`
- …依此类推

**手推第 2 行**（a = "ki"）：
- `dp[2][0] = 2`（初始化：删除 k、i）
- `dp[2][1]`：`a[1]='i'` vs `b[0]='s'` 不同 → `1 + min(dp[1][1]=1, dp[2][0]=2, dp[1][0]=1) = 2`
- `dp[2][2]`：`a[1]='i'` vs `b[1]='i'` **相同** → `dp[1][1] = 1`（直接抄左上）
- `dp[2][3]`：`a[1]='i'` vs `b[2]='t'` 不同 → `1 + min(dp[1][3]=3, dp[2][2]=1, dp[1][2]=2) = 2`
- …依此类推

**关键观察**：
- 当 `a[i-1] == b[j-1]` 时（如第 2 行第 2 列的 'i'），直接**抄左上角**，不增加成本
- 每一格都依赖左上、上、左三个邻居，体现了子问题的分解

完整的 7×8 表最终得到 `dp[6][7] = 3`（对应"kitten"全部 6 字符 → "sitting"全部 7 字符）。

In [ ]:
def edit_distance(a, b) -> int:
    """字符级别 Levenshtein 编辑距离，纯 Python 动态规划。

    同样接受词列表（任何可索引序列），因此词级别 WER 也可以复用此函数。
    这个设计（只依赖 len() 和索引操作）使得同一份 DP 代码能处理字符、词、
    甚至任意对象的序列，体现了算法与数据表示的分离——是正确的代码抽象。

    提示：
      dp[i][j] = 将 a[:i] 变成 b[:j] 的最小编辑次数
               （即 a 的前 i 个元素 vs b 的前 j 个元素）
      边界：dp[i][0] = i（全删），dp[0][j] = j（全插）
      递推：a[i-1]==b[j-1] → dp[i-1][j-1]（不操作）
             否则 → 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
                      （删除、插入、替换取最小）
    """
    # TODO: 按上方递推公式填写 DP 表，返回 dp[m][n]
    raise NotImplementedError("你的实现：参考上方 DP 递推公式")

<details>
<summary>📖 <b>参考答案（SOLUTION）</b> —— 完成任务 1后再展开查看（此格为 markdown，不会执行、不会覆盖你上面的实现）</summary>

```python
def edit_distance(a, b) -> int:
    """字符级别 Levenshtein 编辑距离，纯 Python 动态规划。
    同样接受词列表（任何可索引序列）。
    """
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n]
```

</details>


In [ ]:
# 验证
try:
    assert edit_distance('', '') == 0
    assert edit_distance('abc', '') == 3
    assert edit_distance('', 'abc') == 3
    assert edit_distance('kitten', 'sitting') == 3   # 经典例子
    assert edit_distance('hello', 'hello') == 0
    print('✅ 字符级别编辑距离验证通过')
except (NotImplementedError, TypeError):
    print('⚠️  edit_distance 尚未实现，请完成任务 1')


## ✏️ 任务 2：实现词级别 WER

**签名**：
```python
def wer(reference: str, hypothesis: str) -> float:
    # reference/hypothesis 是空格分隔的字符串
    # 先 .split()，再调 edit_distance(词列表, 词列表)
    # 返回 edit_distance / len(ref_words)
    # reference 为空：hypothesis 也为空返回 0.0，否则返回 float('inf')
```

**关键设计**：直接复用 `edit_distance()`，只是把"字符"改成"词"。

**验收标准**：

| reference | hypothesis | WER |
|---|---|---|
| `'hello world'` | `'hello world'` | `0.0` |
| `'hello world'` | `'hello'` | `0.5` |
| `'a b c'` | `'a x c'` | `0.333...` |
| `''` | `''` | `0.0` |
| `''` | `'anything'` | `inf`（无参考词，无法归一化）|

**为什么 WER 可以 > 1.0？**

如果语音识别系统输出的词远多于参考词（比如重复或幻听），编辑距离会很大。具体例子：
- **reference** = `'hello'`（1 词）
- **hypothesis** = `'i am very sorry for this mistake'`（7 词）
- **编辑距离** = 需要删除"hello"（1 次删除）+ 插入 7 个输出词的等价成本，或简单地：编辑距离 = 7（将 7 个输出词中最多 1 个匹配"hello"，其余全是错）
- **WER** = 7 / 1 = **7.0**（远超 100%）

这在现实中罕见（系统不会好到这程度），但理论允许；反之，WER 接近 0% 表示系统很少出错。

**注意**：`len(reference.split())` 就是 N（参考词数）。

In [ ]:
def wer(reference: str, hypothesis: str) -> float:
    """Word Error Rate = 词级别编辑距离 / 参考词数。

    提示：
      1. 将 reference / hypothesis 小写并 .split() 得到词列表
      2. 处理 ref_words 为空的边界情况（见已提供的代码）
      3. 调用 edit_distance(ref_words, hyp_words) 复用 DP——
         edit_distance 对任意可索引序列都适用，无需重写 DP 循环！
      4. 返回 距离 / len(ref_words)
    """
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else float('inf')
    # TODO: 调用 edit_distance(ref_words, hyp_words) 计算词级别距离
    raise NotImplementedError("你的实现：调用 edit_distance(ref_words, hyp_words)")


<details>
<summary>📖 <b>参考答案（SOLUTION）</b> —— 完成任务 2后再展开查看（此格为 markdown，不会执行、不会覆盖你上面的实现）</summary>

```python
# 关键设计：edit_distance 接受任意可索引序列，词列表同样适用；
# 直接复用，无需重写 DP 循环——这是正确的代码抽象。

def wer(reference: str, hypothesis: str) -> float:
    """Word Error Rate = 词级别编辑距离 / 参考词数。"""
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else float('inf')
    # 复用 edit_distance：字符级和词级使用同一 DP 骨架，只换输入序列类型
    return edit_distance(ref_words, hyp_words) / len(ref_words)
```

</details>


In [ ]:
# 验证
try:
    assert wer('hello world', 'hello world') == 0.0
    assert wer('hello world', 'hello') == 0.5        # 1 deletion / 2 words
    assert abs(wer('a b c', 'a b c d') - 1/3) < 1e-9 # 1 insertion / 3 ref words

    ref = 'the cat sat on the mat'
    hyp = 'the cat set on mat'
    print(f'ref: {ref}')
    print(f'hyp: {hyp}')
    print(f'WER = {wer(ref, hyp):.2%}')  # sat->set (S), deleted 'the' (D) = 2/6
    print('✅ WER 验证通过')
except (NotImplementedError, TypeError):
    print('⚠️  wer 尚未实现，请完成任务 2')


## 对齐回溯：可视化每个错误

回溯 dp 表可以知道每个错误是 S（替换）/I（插入）/D（删除）。

**关于多条最优路径**：当 DP 表中多个邻居都能给出 `min()` 的值时，就存在**多条编辑距离相同的对齐路径**。代码中用 `if...elif...elif...else` 按顺序判断，选择**第一个匹配的操作**（优先级：M > S > I > D），所以输出是某一条最优路径而非全部。这对 WER 计算**没有影响**——WER 只关心最小编辑距离这个数字，不关心具体对齐方式；但在人工审视错误时，不同的对齐路径可能给出不同的语言学解释。如果需要穷举所有最优路径，可以修改回溯逻辑为递归遍历所有同价邻居，但通常不必。

In [ ]:
def alignment(ref_str, hyp_str):
    """打印词级别对齐路径，标注 M/S/I/D。"""
    a, b = ref_str.split(), hyp_str.split()
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    ops, i, j = [], m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and a[i-1] == b[j-1]:
            ops.append(('M', a[i-1], b[j-1])); i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == 1 + dp[i-1][j-1]:
            ops.append(('S', a[i-1], b[j-1])); i -= 1; j -= 1
        elif j > 0 and dp[i][j] == 1 + dp[i][j-1]:
            ops.append(('I', '-', b[j-1])); j -= 1
        else:
            ops.append(('D', a[i-1], '-')); i -= 1
    print(f'{"Op":2s}  {"REF":12s}  {"HYP":12s}')
    print('-' * 30)
    for op, r, h in reversed(ops):
        marker = '  ' if op == 'M' else '**'
        print(f'{marker}{op:2s}  {r:12s}  {h:12s}{marker}')

alignment('the cat sat on the mat', 'the cat set on mat')

## 本课收束

| 概念 | 要记住的 |
|---|---|
| 编辑距离 | 3 种操作，DP 表 (m+1)×(n+1) |
| WER | = 词级编辑距离 / 参考词数，可以 > 1.0 |
| 回溯 | 从 dp[m][n] 往左上角走，还原对齐路径 |
| L68 | CTC 对齐——同样是"所有对齐路径"思想，但面对的是连续帧 |

下一课（L68）将用 CTC 对齐代替手动标注：`ctc_alignment` 把声学帧序列与目标词序列自动对齐，服务于 CTC 系列模型（如 wav2vec 2.0、DeepSpeech）的训练目标。

L69 的 CTC 前向算法也是同一种“填表递推”思想，只是把最小化编辑代价换成 log 概率求和。

## Aurora 连接

本课实现的两个函数对应 `src/aurora/speech/` 生产模块中的 ASR 评估工具：

| 本课函数 | 生产路径 | 说明 |
|---|---|---|
| `edit_distance(a, b)` | `src/aurora/speech/metrics.py::edit_distance` | 泛化版，接受任意序列 |
| `wer(reference, hypothesis)` | `src/aurora/speech/metrics.py::wer` | 委托给 `edit_distance`，无重复 DP |

生产实现的额外特性：
- **O(n) 空间优化**：只需保留相邻两行 `dp[i-1]` 和 `dp[i]`，
  当只需要距离值（不需要回溯路径）时，内存从 O(m×n) 降为 O(n)。
- **批量 WER**：接受多句 `List[str]`，对语料集整体汇报错误率。

```python
# 查看生产模块
from aurora.speech.metrics import edit_distance, wer
print(wer('the cat sat on the mat', 'the cat set on mat'))  # 0.333...
```


## ✏️ 闭卷推导检查格 — Edit Distance DP 表格

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**："cat" → "cut" 的 Levenshtein 距离

1. 写出递推公式（三种操作：插入、删除、替换）
2. 手填完整的 DP 表格（4×4，含边界行/列）
3. 从表格回溯，写出一条最短编辑路径

|   |   | c | u | t |
|---|---|---|---|---|
|   | 0 | 1 | 2 | 3 |
| c | 1 |   |   |   |
| a | 2 |   |   |   |
| t | 3 |   |   |   |

（在此处填表并写路径...）

In [ ]:
# 验证：你填的表格与算法结果一致
import sys; sys.path.insert(0, 'src')

def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n], dp

dist, table = levenshtein("cat", "cut")
assert dist == 1, f"期望距离 1，得到 {dist}"
print("DP 表格（行=cat，列=cut）：")
header = "    " + "  ".join([" "] + list("cut"))
print(header)
for i, row_label in enumerate(" cat"):
    print(f"  {row_label} " + "  ".join(str(table[i][j]) for j in range(4)))
print(f"\n✅ Edit distance('cat','cut') = {dist}")

In [ ]:
# ✏️ 本课自评
l67_review = {
    "levenshtein_dp_memorized":  None,  # 能默写DP转移方程（相同→不变，不同→1+min(删/插/替)）？True/False
    "edit_distance_impl":        None,  # edit_distance()实现正确，5个断言全通过？True/False
    "wer_impl":                  None,  # wer()实现正确，词级版本复用edit_distance？True/False
    "dp_table_traced":           None,  # 白板手推"cat"→"cut"的完整4×4 DP表（含边界行/列）？True/False
    "whiteboard_passed":         None,  # 白板挑战"cat"→"cut"推导格通关？True/False
}

unfilled = [k for k, v in l67_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l67_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L67 全部通关！进入 L68：CTC 对齐原理')

---

→ **下一课**　[L68 · CTC 对齐原理](L68_ctc_alignment.ipynb)

> 下节课将学习 **CTC 对齐原理**：blank 符号、单调路径与标签折叠。